In [1]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from rdkit import rdBase
from torch.utils.data import DataLoader

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.bilstm import BiLSTMNetwork
from dataset.smiles import PAC5SMILESDataset, SMILESTokenizer, get_collate_fn

rdBase.DisableLog("rdApp.*")
torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
dataset = "PAC5-3"
model_path_template = f"../models/bilstm_1.1M-smiles-[{dataset}]-fold_{{fold}}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = f"bilstm_1.1M-smiles-[{dataset}]-fold_{{fold}}"
num_tasks = len(target_properties)
loss_weights = torch.ones(num_tasks, device=device)

# Build global SMILES vocabulary
print("Building global SMILES vocabulary...")
all_smiles = []
for fold in range(1, k_folds + 1):
    train_df = pd.read_csv(model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv")
    test_df = pd.read_csv(model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv")
    all_smiles.extend(train_df["smiles"].tolist())
    all_smiles.extend(test_df["smiles"].tolist())

tokenizer = SMILESTokenizer(all_smiles)
collate_fn = get_collate_fn(tokenizer)
print(f"Vocabulary size: {tokenizer.vocab_size}")

W0930 11:42:45.927000 14716 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda
Building global SMILES vocabulary...
Vocabulary size: 32


In [2]:
# -----------------------------------------------------------------------------
# 1. Training Loop
# -----------------------------------------------------------------------------
fold_results = []

for fold in range(1, k_folds + 1):
    if fold != 5:
        continue
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    val_path = model_dataset_dir + f"/{dataset}/fold_{fold}_val.csv"

    train_dataset = PAC5SMILESDataset(train_path, properties=target_properties)
    val_dataset = PAC5SMILESDataset(val_path, properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    val_dataset.targets = torch.tensor(
        scaler.transform(val_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        pin_memory=True,
        collate_fn=collate_fn,
        num_workers=8,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        pin_memory=True,
        collate_fn=collate_fn,
        num_workers=8,
    )

    core_model = BiLSTMNetwork(vocab_size=tokenizer.vocab_size, num_tasks=num_tasks)
    lightning_model = LightningModule(core_model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=10,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path,
        model=core_model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    del train_dataset
    del val_dataset
    del train_loader
    del val_loader
    del core_model
    del lightning_model
    del trainer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 5/5 ]======


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precis

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ BiLSTMNetwork │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss       │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict    │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict    │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict    │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict    │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict    │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict    │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict    │      0 │ train │     0 │
└───┴────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.371                                                                      
Modules in train mode: 65                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 0.22604 | xtb_homo Val RMSD: 0.18825 | xtb_lumo Val RMSD: 0.21490 | xtb_hl_gap Val RMSD: 0.27183 | pm7_homo Val RMSD: 0.19920 | pm7_lumo Val RMSD: 0.17731 | pm7_hl_gap Val RMSD: 0.25517 | pm7_hof Val RMSD: 103.17652
Epoch 002 | Scaled Train Loss: 0.34479 | Scaled Val Loss: 0.17865 | xtb_homo Val RMSD: 0.16917 | xtb_lumo Val RMSD: 0.18558 | xtb_hl_gap Val RMSD: 0.24421 | pm7_homo Val RMSD: 0.17864 | pm7_lumo Val RMSD: 0.15513 | pm7_hl_gap Val RMSD: 0.22877 | pm7_hof Val RMSD: 89.00734
Epoch 003 | Scaled Train Loss: 0.20298 | Scaled Val Loss: 0.15399 | xtb_homo Val RMSD: 0.15621 | xtb_lumo Val RMSD: 0.17230 | xtb_hl_gap Val RMSD: 0.22570 | pm7_homo Val RMSD: 0.16722 | pm7_lumo Val RMSD: 0.14418 | pm7_hl_gap Val RMSD: 0.21438 | pm7_hof Val RMSD: 79.46194
Epoch 004 | Scaled Train Loss: 0.17055 | Scaled Val Loss: 0.14664 | xtb_homo Val RMSD: 0.14827 | xtb_lumo Val RMSD: 0.16814 | xtb_hl_gap Val RMSD: 0.22476 | pm7_homo Val RMSD: 0.15951 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00198 | Scaled Val Loss: 0.01987 | xtb_homo Val RMSD: 0.05654 | xtb_lumo Val RMSD: 0.05802 | xtb_hl_gap Val RMSD: 0.07609 | pm7_homo Val RMSD: 0.06953 | pm7_lumo Val RMSD: 0.05469 | pm7_hl_gap Val RMSD: 0.08538 | pm7_hof Val RMSD: 18.30597
Best scaled validation loss for fold 5: 0.0197

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0197]
Average Scaled Loss: 0.0197 ± 0.0000


In [2]:
# -----------------------------------------------------------------------------
# 2. Evaluation Loop
# -----------------------------------------------------------------------------

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
    "test": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
}

for fold in range(1, k_folds + 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"

    train_dataset = PAC5SMILESDataset(train_path, properties=target_properties)
    test_dataset = PAC5SMILESDataset(test_path, properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.targets.numpy())

    train_dataset.targets = torch.tensor(
        scaler.transform(train_dataset.targets.numpy()), dtype=torch.float32
    )
    test_dataset.targets = torch.tensor(
        scaler.transform(test_dataset.targets.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=12,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=256,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=12,
    )

    core_model = BiLSTMNetwork(vocab_size=tokenizer.vocab_size, num_tasks=num_tasks)
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    core_model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(core_model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        for metric in metric_keys:
            all_results["train"][prop][metric].append(
                train_results[f"test_{metric}_{prop}"]
            )
            all_results["test"][prop][metric].append(
                test_results[f"test_{metric}_{prop}"]
            )

print("\n======[ Cross-validation results (mean ± std) ]======")

for property_name in target_properties:
    print(f"\n======[ {property_name} ]======")

    train_statistics = {
        metric: (
            np.mean(all_results["train"][property_name][metric]),
            np.std(all_results["train"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {train_statistics['mae'][0]:.4f} ± {train_statistics['mae'][1]:.4f}\t"
        f"RMSD: {train_statistics['rmsd'][0]:.4f} ± {train_statistics['rmsd'][1]:.4f}\t"
        f"Max: {train_statistics['max'][0]:.4f} ± {train_statistics['max'][1]:.4f}\t"
        f"Min: {train_statistics['min'][0]:.4f} ± {train_statistics['min'][1]:.4f}\t"
        f"R^2: {train_statistics['r2'][0]:.4f} ± {train_statistics['r2'][1]:.4f}"
    )

    test_statistics = {
        metric: (
            np.mean(all_results["test"][property_name][metric]),
            np.std(all_results["test"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {test_statistics['mae'][0]:.4f} ± {test_statistics['mae'][1]:.4f}\t"
        f"RMSD: {test_statistics['rmsd'][0]:.4f} ± {test_statistics['rmsd'][1]:.4f}\t"
        f"Max: {test_statistics['max'][0]:.4f} ± {test_statistics['max'][1]:.4f}\t"
        f"Min: {test_statistics['min'][0]:.4f} ± {test_statistics['min'][1]:.4f}\t"
        f"R^2: {test_statistics['r2'][0]:.4f} ± {test_statistics['r2'][1]:.4f}"
    )


======[ Evaluating fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.



======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0164	RMSD: 0.0218	Max: 0.2662	Min: 0.0000	R^2: 0.9975
 + Test -> MAE: 0.0323	RMSD: 0.0536	Max: 1.1120	Min: 0.0000	R^2: 0.9846
======[ xtb_lumo ]======
 + Train -> MAE: 0.0184	RMSD: 0.0241	Max: 0.4379	Min: 0.0000	R^2: 0.9980
 + Test -> MAE: 0.0356	RMSD: 0.0578	Max: 1.2011	Min: 0.0000	R^2: 0.9882
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0194	RMSD: 0.0254	Max: 0.4219	Min: 0.0000	R^2: 0.9962
 + Test -> MAE: 0.0454	RMSD: 0.0774	Max: 1.8231	Min: 0.0000	R^2: 0.9647
======[ pm7_homo ]======
 + Train -> MAE: 0.0210	RMSD: 0.0284	Max: 0.3847	Min: 0.0000	R^2: 0.9969
 + Test -> MAE: 0.0419	RMSD: 0.0662	Max: 1.2002	Min: 0.0000	R^2: 0.9830
======[ pm7_lumo ]======
 + Train -> MAE: 0.0166	RMSD: 0.0219	Max: 0.3051	Min: 0.0000	R^2: 0.9961
 + Test -> MAE: 0.0347	RMSD: 0.0557	Max: 2.8334	Min: 0.0000	R^2: 0.9745
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0216	RMSD: 0.0294	Max: 0.4261	Min: 0.0000	R^2: 0.9958
 + Test -> 

...


======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0162	RMSD: 0.0214	Max: 0.2268	Min: 0.0000	R^2: 0.9975
 + Test -> MAE: 0.0332	RMSD: 0.0553	Max: 1.1255	Min: 0.0000	R^2: 0.9836
======[ xtb_lumo ]======
 + Train -> MAE: 0.0185	RMSD: 0.0243	Max: 0.3285	Min: 0.0000	R^2: 0.9979
 + Test -> MAE: 0.0361	RMSD: 0.0586	Max: 1.5307	Min: 0.0000	R^2: 0.9879
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0183	RMSD: 0.0240	Max: 0.3661	Min: 0.0000	R^2: 0.9966
 + Test -> MAE: 0.0458	RMSD: 0.0777	Max: 1.8882	Min: 0.0000	R^2: 0.9647
======[ pm7_homo ]======
 + Train -> MAE: 0.0208	RMSD: 0.0281	Max: 0.3785	Min: 0.0000	R^2: 0.9969
 + Test -> MAE: 0.0428	RMSD: 0.0674	Max: 1.4641	Min: 0.0000	R^2: 0.9824
======[ pm7_lumo ]======
 + Train -> MAE: 0.0162	RMSD: 0.0214	Max: 0.2807	Min: 0.0000	R^2: 0.9963
 + Test -> MAE: 0.0350	RMSD: 0.0556	Max: 1.5195	Min: 0.0000	R^2: 0.9749
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0215	RMSD: 0.0291	Max: 0.3946	Min: 0.0000	R^2: 0.9959
 + Test -> 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0163	RMSD: 0.0215	Max: 0.2182	Min: 0.0000	R^2: 0.9975
 + Test -> MAE: 0.0329	RMSD: 0.0548	Max: 1.4138	Min: 0.0000	R^2: 0.9839
======[ xtb_lumo ]======
 + Train -> MAE: 0.0179	RMSD: 0.0235	Max: 0.3774	Min: 0.0000	R^2: 0.9981
 + Test -> MAE: 0.0354	RMSD: 0.0581	Max: 1.0817	Min: 0.0000	R^2: 0.9882
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0182	RMSD: 0.0239	Max: 0.3697	Min: 0.0000	R^2: 0.9967
 + Test -> MAE: 0.0454	RMSD: 0.0772	Max: 1.8623	Min: 0.0000	R^2: 0.9649
======[ pm7_homo ]======
 + Train -> MAE: 0.0206	RMSD: 0.0279	Max: 0.3748	Min: 0.0000	R^2: 0.9970
 + Test -> MAE: 0.0424	RMSD: 0.0667	Max: 1.8954	Min: 0.0000	R^2: 0.9828
======[ pm7_lumo ]======
 + Train -> MAE: 0.0162	RMSD: 0.0213	Max: 0.2706	Min: 0.0000	R^2: 0.9963
 + Test -> MAE: 0.0353	RMSD: 0.0558	Max: 1.8485	Min: 0.0000	R^2: 0.9747
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0212	RMSD: 0.0287	Max: 0.4111	Min: 0.0000	R^2: 0.9960
 + Test -> 